# 🌐 Обучение модели прогноза RPS

Загрузка датасета по endpoint-ам, обучение Ridge per-handler (горизонт 5 мин) и сохранение модели.


## 1. Настройка окружения

In [1]:
import warnings; warnings.filterwarnings('ignore')

import json
from pathlib import Path
import os

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import joblib

FREQ_SEC     = 15
HORIZONS_MIN = [1, 5, 10, 15, 30, 45, 60]
print('✅ Окружение готово')


✅ Окружение готово


## 2. Загрузка и обзор датасета RPS


In [2]:
RPS_FILES = [
    '../data/users__rps.csv',
    '../data/accounts__rps.csv',
    '../data/transfers__rps.csv',
]
dfs = []
for f in RPS_FILES:
    if os.path.exists(f):
        d = pd.read_csv(f)
        dfs.append(d)
        print(f"Loaded {f}: {d.shape}")

raw = pd.concat(dfs, ignore_index=True)
raw['datetime'] = pd.to_datetime(raw['timestamp'], unit='s')
raw = raw.sort_values(['handler', 'datetime']).reset_index(drop=True)

HANDLERS = sorted(raw['handler'].unique().tolist())
print(f"\nHandlers: {HANDLERS}")

# Каждый handler — собственный ряд с шагом 15 с
series_by_handler: dict[str, pd.Series] = {}
for h in HANDLERS:
    sub = raw[raw['handler'] == h].drop_duplicates('timestamp')
    sub = sub.set_index('datetime')['value']
    full_idx = pd.date_range(sub.index.min(), sub.index.max(), freq='15s')
    sub = sub.reindex(full_idx).interpolate('time').fillna(0.0).clip(lower=0.0)
    sub.index.name = 'datetime'
    series_by_handler[h] = sub
    print(f"  {h:32s}  rows={len(sub):>7}  "
          f"mean={sub.mean():.2f}  max={sub.max():.2f}")

Loaded ../data/users__rps.csv: (30134, 4)
Loaded ../data/accounts__rps.csv: (30134, 4)
Loaded ../data/transfers__rps.csv: (30134, 4)

Handlers: ['/accounts/', '/transfers/', '/users/']
  /accounts/                        rows=  30134  mean=15.00  max=49.02
  /transfers/                       rows=  30134  mean=18.00  max=42.59
  /users/                           rows=  30134  mean=0.20  max=0.55


## 3. Признаки для Ridge (общие на все handler-ы)


In [3]:
def make_features(series, extra_cols: dict | None = None):
    """
    Признаки для Ridge — копия `make_features` из cpu_walkforward.ipynb.
    extra_cols (dict[str, pd.Series]) — дополнительные колонки (например, one-hot
    handler-а для RPS); индексируются тем же индексом, что и series.
    """
    f = pd.DataFrame(index=series.index)
    for lag in [1, 2, 4, 8, 16, 32, 60, 120, 240]:
        f[f'lag_{lag}'] = series.shift(lag)
    for w in [4, 20, 60, 120, 240]:
        f[f'roll_mean_{w}'] = series.shift(1).rolling(w).mean()
        f[f'roll_std_{w}']  = series.shift(1).rolling(w).std()
    for span in [4, 20, 60]:
        f[f'ewm_{span}'] = series.shift(1).ewm(span=span).mean()
    f['hour_sin']   = np.sin(2*np.pi*series.index.hour/24)
    f['hour_cos']   = np.cos(2*np.pi*series.index.hour/24)
    f['minute_sin'] = np.sin(2*np.pi*series.index.minute/60)
    f['minute_cos'] = np.cos(2*np.pi*series.index.minute/60)
    f['diff_1']  = series.diff(1).shift(1)
    f['diff_20'] = series.diff(20).shift(1)
    if extra_cols:
        for k, v in extra_cols.items():
            f[k] = v.reindex(f.index).values
    return f

BASE_FEATURE_COLS = (
    [f'lag_{l}'       for l in [1, 2, 4, 8, 16, 32, 60, 120, 240]]
    + [f'roll_mean_{w}' for w in [4, 20, 60, 120, 240]]
    + [f'roll_std_{w}'  for w in [4, 20, 60, 120, 240]]
    + [f'ewm_{s}'       for s in [4, 20, 60]]
    + ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'diff_1', 'diff_20']
)
LAGS         = [1, 2, 4, 8, 16, 32, 60, 120, 240]
ROLL_WINDOWS = [4, 20, 60, 120, 240]
EWM_SPANS    = [4, 20, 60]
MIN_HISTORY_POINTS = max(max(LAGS), max(ROLL_WINDOWS)) + 1   # 241
ALPHA_RIDGE  = 10.0

print(f"Признаков без extra: {len(BASE_FEATURE_COLS)}")

Признаков без extra: 28


## 4. Обучение модели и сохранение

Фиксируем горизонт 5 мин. Для каждого handler-а — собственная пара (Ridge, StandardScaler), всё кладётся в один `joblib`-bundle.


In [4]:
FINAL_HORIZON_MIN   = 5
FINAL_HORIZON_STEPS = int(FINAL_HORIZON_MIN * 60 / FREQ_SEC)
FEATURE_COLS = list(BASE_FEATURE_COLS)

# Для каждого handler-а — отдельный Ridge + Scaler.
# RPS-ряды очень разные по масштабу (см. сводку выше), поэтому общий scaler
# был бы плохой идеей. Отдельные модели:
#   * сохраняют единый набор фич (совместимость с RidgeFeatureBuilder),
#   * корректно учитывают per-endpoint масштаб через свой StandardScaler.

handler_models: dict[str, dict] = {}
per_handler_metrics: dict[str, dict] = {}

for h_name, series in series_by_handler.items():
    X_full = make_features(series)[FEATURE_COLS]
    y_full = series.shift(-FINAL_HORIZON_STEPS).rename('target')
    data_full = pd.concat([X_full, y_full], axis=1).dropna()
    if len(data_full) < 200:
        print(f"⚠️  {h_name}: слишком мало точек ({len(data_full)}), пропускаем")
        continue

    X_train = data_full[FEATURE_COLS].values
    y_train = data_full['target'].values

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)
    ridge = Ridge(alpha=ALPHA_RIDGE).fit(Xs, y_train)
    y_pred_in = ridge.predict(Xs)

    _mask_nz = y_train != 0
    in_sample = dict(
        MSE=float(mean_squared_error(y_train, y_pred_in)),
        MAE=float(mean_absolute_error(y_train, y_pred_in)),
        MAPE=float(np.mean(np.abs((y_pred_in[_mask_nz] - y_train[_mask_nz]) / y_train[_mask_nz])) * 100
               if _mask_nz.any() else float('nan')),
        WAPE=float(np.sum(np.abs(y_pred_in - y_train)) / np.sum(np.abs(y_train)) * 100
               if np.sum(np.abs(y_train)) > 0 else float('nan')),
    )
    per_handler_metrics[h_name] = in_sample
    print(f"{h_name:32s}  MSE={in_sample['MSE']:.4f}  MAE={in_sample['MAE']:.3f}  MAPE={in_sample['MAPE']:.2f}%  WAPE={in_sample['WAPE']:.2f}%")

    handler_models[h_name] = dict(model=ridge, scaler=scaler)

# ── Сохранение ─────────────────────────────────────────────────────────
MODELS_DIR  = Path('../models'); MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH  = MODELS_DIR / 'rps_forecast_model.joblib'
CONFIG_PATH = MODELS_DIR / 'rps_model_config.json'

handler_mapping = {h: i for i, h in enumerate(HANDLERS)}

bundle = {
    # Совместимость с RidgePredictor: общий model/scaler = модель первого handler-а
    # как дефолтная (если handler неизвестен инференсу). Основное хранилище —
    # 'handlers' с per-endpoint моделями.
    'model':  next(iter(handler_models.values()))['model']  if handler_models else None,
    'scaler': next(iter(handler_models.values()))['scaler'] if handler_models else None,
    'feature_cols': FEATURE_COLS,
    'handlers': handler_models,
    'handler_mapping': handler_mapping,
    'meta': {
        'model_type': 'ridge',
        'metric_type': 'rps',
        'per_handler': True,
        'forecast_horizon_steps': FINAL_HORIZON_STEPS,
        'forecast_horizon_seconds': FINAL_HORIZON_STEPS * FREQ_SEC,
        'scrape_interval_sec': FREQ_SEC,
        'lags': LAGS,
        'rolling_windows': ROLL_WINDOWS,
        'ewm_spans': EWM_SPANS,
        'min_history_points': MIN_HISTORY_POINTS,
        'alpha': ALPHA_RIDGE,
    },
}
joblib.dump(bundle, MODEL_PATH)
print(f"\n✅ Модель сохранена: {MODEL_PATH.resolve()}")

config_json = {
    'model_type': 'ridge',
    'metric_type': 'rps',
    'per_handler': True,
    'feature_cols': FEATURE_COLS,
    'categorical_features': [],
    'handler_mapping': handler_mapping,
    'forecast_horizon_steps': FINAL_HORIZON_STEPS,
    'forecast_horizon_seconds': FINAL_HORIZON_STEPS * FREQ_SEC,
    'scrape_interval_sec': FREQ_SEC,
    'lags': LAGS,
    'rolling_windows': ROLL_WINDOWS,
    'ewm_spans': EWM_SPANS,
    'min_history_points': MIN_HISTORY_POINTS,
    'alpha': ALPHA_RIDGE,
    'per_handler_in_sample': per_handler_metrics,
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config_json, f, indent=2, ensure_ascii=False)
print(f"✅ Конфиг сохранён:  {CONFIG_PATH.resolve()}")

/accounts/                        MSE=3.3407  MAE=0.922  MAPE=11.27%  WAPE=6.10%
/transfers/                       MSE=4.8918  MAE=1.165  MAPE=11.78%  WAPE=6.42%
/users/                           MSE=0.0019  MAE=0.030  MAPE=30.62%  WAPE=14.79%

✅ Модель сохранена: /Users/anastasiagusak/Documents/4 курс/MLPredictor/models/rps_forecast_model.joblib
✅ Конфиг сохранён:  /Users/anastasiagusak/Documents/4 курс/MLPredictor/models/rps_model_config.json
